In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from matplotlib.gridspec import GridSpec

# Project modules
from src.data_loader import load_all
from src.expansion_engine import build_branch_features, score_branches, generate_recommendation, run_expansion_analysis
from src.staffing_engine import run_staffing_analysis

# Styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
COLORS = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3B1F2B']
BRANCH_COLORS = {
    'Conut': COLORS[0],
    'Conut - Tyre': COLORS[1],
    'Conut Jnah': COLORS[2],
    'Main Street Coffee': COLORS[3],
}

print('Libraries loaded successfully.')

In [ ]:
# Load all datasets
dfs = load_all()
monthly   = dfs['monthly_sales']
menu      = dfs['avg_sales_by_menu']
tax       = dfs['tax_summary']
att       = dfs['attendance']
items     = dfs['items_sales']
orders    = dfs['customer_orders']

print('Dataset shapes:')
for k, v in dfs.items():
    print(f'  {k:25s}: {v.shape}')

---
## Part 1 — Expansion Feasibility

In [ ]:
# ── Figure 1: Monthly Revenue Trajectories ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: absolute revenue per branch per month
ax = axes[0]
for branch in monthly['branch'].unique():
    bm = monthly[monthly['branch'] == branch].sort_values('month')
    color = BRANCH_COLORS.get(branch, 'gray')
    ax.plot(bm['month_name'], bm['revenue'] / 1e9, marker='o', label=branch,
            color=color, linewidth=2.5, markersize=7)

ax.set_title('Monthly Revenue by Branch (Aug–Dec 2025)', fontsize=13, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue (Billions, scaled units)')
ax.legend(fontsize=9)
ax.tick_params(axis='x', rotation=20)

# Right: revenue indexed to first observed month (growth index)
ax2 = axes[1]
for branch in monthly['branch'].unique():
    bm = monthly[monthly['branch'] == branch].sort_values('month')
    if len(bm) < 2:
        continue
    base = bm['revenue'].iloc[0]
    indexed = (bm['revenue'] / base) * 100
    color = BRANCH_COLORS.get(branch, 'gray')
    ax2.plot(bm['month_name'], indexed, marker='s', label=branch,
             color=color, linewidth=2.5, markersize=7)

ax2.axhline(100, color='black', linestyle='--', alpha=0.4, label='Baseline (100)')
ax2.set_title('Revenue Growth Index (Base = 100 at first month)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Month')
ax2.set_ylabel('Index')
ax2.legend(fontsize=9)
ax2.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig('expansion_revenue_trends.png', dpi=150, bbox_inches='tight')
plt.show()
print('Fig 1 saved: expansion_revenue_trends.png')

In [ ]:
# ── Figure 2: Branch Feature Radar / Spider Chart ───────────────────────────
from matplotlib.patches import FancyArrowPatch

features = build_branch_features()
scored   = score_branches(features)

# Radar chart: normalise 0-1 for display
radar_cols = ['monthly_growth_rate', 'revenue_momentum', 'total_customers',
              'avg_revenue_per_customer', 'delivery_pct', 'repeat_customer_pct']
radar_labels = ['Monthly\nGrowth', 'Revenue\nMomentum', 'Customer\nVolume',
                'Rev/Customer', 'Delivery\nShare', 'Repeat\nCustomers']

# Normalise
norm_df = features[radar_cols].copy()
for col in radar_cols:
    mn, mx = norm_df[col].min(), norm_df[col].max()
    norm_df[col] = (norm_df[col] - mn) / (mx - mn) if mx > mn else 0.5

N = len(radar_cols)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)

for branch in norm_df.index:
    vals = norm_df.loc[branch, radar_cols].tolist()
    vals += vals[:1]
    color = BRANCH_COLORS.get(branch, 'gray')
    ax.plot(angles, vals, 'o-', linewidth=2, label=branch, color=color)
    ax.fill(angles, vals, alpha=0.12, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, fontsize=11)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['25%', '50%', '75%', '100%'], fontsize=8)
ax.set_title('Branch Performance Radar\n(Normalised Scores)', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=10)

plt.tight_layout()
plt.savefig('expansion_radar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Fig 2 saved: expansion_radar.png')

In [ ]:
# ── Figure 3: Composite Expansion Score Bar Chart ───────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

score_data = scored['expansion_score'].sort_values(ascending=True)
colors_bar = [BRANCH_COLORS.get(b, 'gray') for b in score_data.index]

bars = ax.barh(score_data.index, score_data.values, color=colors_bar,
               edgecolor='white', linewidth=0.8, height=0.55)

for bar, val in zip(bars, score_data.values):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}', va='center', fontsize=12, fontweight='bold')

ax.set_xlabel('Expansion Attractiveness Score (0-100)', fontsize=11)
ax.set_title('Branch Expansion Score\n(Higher = Better Template to Replicate)',
             fontsize=13, fontweight='bold')
ax.set_xlim(0, 115)
ax.axvline(75, color='green', linestyle='--', alpha=0.6, label='GO threshold (75)')
ax.axvline(40, color='orange', linestyle='--', alpha=0.6, label='CAUTION threshold (40)')
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('expansion_scores.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 4: Revenue per Customer vs. Total Customers (Bubble Chart) ───────
fig, ax = plt.subplots(figsize=(10, 6))

for branch in features.index:
    x = features.loc[branch, 'total_customers']
    y = features.loc[branch, 'avg_revenue_per_customer'] / 1e9
    s = scored.loc[branch, 'expansion_score'] * 15  # bubble size = score
    c = BRANCH_COLORS.get(branch, 'gray')
    ax.scatter(x, y, s=s, color=c, alpha=0.75, edgecolors='white', linewidths=1.5)
    ax.annotate(branch, (x, y), textcoords='offset points', xytext=(10, 5),
                fontsize=10, fontweight='bold', color=c)

ax.set_xlabel('Total Customers Served (2025)', fontsize=11)
ax.set_ylabel('Avg Revenue per Customer (Billions, scaled)', fontsize=11)
ax.set_title('Customer Volume vs. Revenue per Customer\n'
             '(Bubble size = Expansion Score)', fontsize=13, fontweight='bold')

# Quadrant lines
ax.axvline(features['total_customers'].median(), color='gray', linestyle=':', alpha=0.5)
ax.axhline((features['avg_revenue_per_customer'] / 1e9).median(), color='gray', linestyle=':', alpha=0.5)

ax.text(0.98, 0.98, 'IDEAL ZONE\n(High volume, High spend)', transform=ax.transAxes,
        ha='right', va='top', fontsize=9, color='green',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#e8f5e9', alpha=0.7))

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('expansion_bubble.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Print Full Expansion Report ──────────────────────────────────────────────
from src.expansion_engine import print_report
rec = run_expansion_analysis()
print_report(rec)

---
## Part 2 — Shift Staffing Estimation

In [ ]:
# Run staffing analysis
staffing_result = run_staffing_analysis()
att_stats   = staffing_result['attendance_stats']
shift_prof  = staffing_result['shift_profile']
productivity= staffing_result['productivity_index']
staffing_rec= staffing_result['staffing_recommendations']
anomalies   = staffing_result['anomalies']
att_raw     = staffing_result['raw_attendance']

print('Attendance records loaded:', len(att_raw))
print('\nBranch summary:')
print(att_stats[['unique_employees','total_shifts','avg_shift_hours','avg_daily_staff']].to_string())

In [ ]:
# ── Figure 5: Shift Type Distribution Stacked Bar ───────────────────────────
shift_pct = shift_prof['n_shifts'].unstack(fill_value=0)
shift_pct = shift_pct.div(shift_pct.sum(axis=1), axis=0) * 100

shift_colors = {
    'Morning (06-12)':   '#F18F01',
    'Afternoon (12-17)': '#2E86AB',
    'Evening (17-24)':   '#A23B72',
    'Graveyard (00-06)': '#3B1F2B',
}

fig, ax = plt.subplots(figsize=(10, 5))
bottom = np.zeros(len(shift_pct))
for col in shift_pct.columns:
    color = shift_colors.get(col, 'gray')
    bars = ax.bar(shift_pct.index, shift_pct[col], bottom=bottom,
                  label=col, color=color, edgecolor='white', linewidth=0.8)
    for bar, val in zip(bars, shift_pct[col]):
        if val > 5:
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_y() + bar.get_height() / 2,
                    f'{val:.0f}%', ha='center', va='center',
                    fontsize=9, fontweight='bold', color='white')
    bottom += shift_pct[col].values

ax.set_ylabel('% of Shifts')
ax.set_title('Shift Type Distribution by Branch (December 2025)',
             fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.set_ylim(0, 110)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('staffing_shift_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 6: Daily Staff Coverage Heatmap ──────────────────────────────────
# Build date × branch matrix of concurrent staff
pivot = att_raw.groupby(['date', 'branch'])['emp_id'].nunique().unstack(fill_value=0)

# Convert date strings to datetime for proper ordering
pivot.index = pd.to_datetime(pivot.index, format='%d-%b-%y')
pivot = pivot.sort_index()
pivot.index = pivot.index.strftime('%d-%b')

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(pivot.T, cmap='YlOrRd', ax=ax, linewidths=0.4,
            cbar_kws={'label': 'Staff on Floor'},
            annot=True, fmt='d', annot_kws={'size': 8})
ax.set_title('Daily Staff Coverage per Branch (December 2025)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Branch')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('staffing_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 7: Revenue per Staff-Hour (Productivity) ─────────────────────────
prod_clean = productivity.dropna(subset=['revenue_per_staff_hour'])

fig, ax = plt.subplots(figsize=(9, 5))
colors_prod = [BRANCH_COLORS.get(b, 'gray') for b in prod_clean.index]

bars = ax.bar(prod_clean.index, prod_clean['revenue_per_staff_hour'] / 1e6,
              color=colors_prod, edgecolor='white', linewidth=0.8)

for bar, (branch, row) in zip(bars, prod_clean.iterrows()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f"{row['revenue_per_staff_hour']/1e6:.2f}M",
            ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylabel('Revenue per Staff-Hour (Millions, scaled units)', fontsize=11)
ax.set_title('Staff Productivity Index by Branch\n'
             '(December 2025 — Revenue Generated per Hour of Labour)',
             fontsize=13, fontweight='bold')
ax.tick_params(axis='x', rotation=15)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('staffing_productivity.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 8: Recommended Staff per Shift Slot ──────────────────────────────
shift_order = ['Morning (06-12)', 'Afternoon (12-17)', 'Evening (17-24)', 'Graveyard (00-06)']
rec_matrix = {}
for branch, slots in staffing_rec.items():
    if 'error' not in slots:
        rec_matrix[branch] = {s: slots[s]['recommended_staff'] for s in shift_order if s in slots}

rec_df = pd.DataFrame(rec_matrix, index=shift_order).fillna(0).astype(int)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(shift_order))
width = 0.2

for i, branch in enumerate(rec_df.columns):
    offset = (i - len(rec_df.columns) / 2 + 0.5) * width
    color = BRANCH_COLORS.get(branch, 'gray')
    bars = ax.bar(x + offset, rec_df[branch], width=width,
                  label=branch, color=color, edgecolor='white', linewidth=0.7)

ax.set_xticks(x)
ax.set_xticklabels(shift_order, fontsize=10)
ax.set_ylabel('Recommended Number of Staff')
ax.set_title('Recommended Staff per Shift per Branch',
             fontsize=13, fontweight='bold')
ax.axhline(2, color='red', linestyle='--', alpha=0.5, label='Safety minimum (2)')
ax.legend(fontsize=9)
ax.set_ylim(0, max(rec_df.values.max() + 2, 5))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('staffing_recommendations.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 9: Attendance Anomaly Summary ────────────────────────────────────
if not anomalies.empty:
    print(f'Attendance Anomalies Detected: {len(anomalies)}')
    anom_display = anomalies.copy()
    print(anom_display.to_string(index=False))
    
    fig, ax = plt.subplots(figsize=(8, 4))
    anom_by_branch = anomalies.groupby('branch').size()
    colors_anom = [BRANCH_COLORS.get(b, 'gray') for b in anom_by_branch.index]
    ax.bar(anom_by_branch.index, anom_by_branch.values, color=colors_anom, edgecolor='white')
    ax.set_title('Attendance Anomalies by Branch', fontsize=13, fontweight='bold')
    ax.set_ylabel('Number of Anomalous Records')
    ax.tick_params(axis='x', rotation=15)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.savefig('staffing_anomalies.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No significant anomalies detected.')

In [ ]:
# ── Print Full Staffing Report ───────────────────────────────────────────────
from src.staffing_engine import print_staffing_report
print_staffing_report(staffing_result)

---
## Part 3 — Executive Summary

### Expansion Decision
| Signal | Finding |
|--------|---------|
| Network growth | +56% avg monthly revenue growth (Aug-Dec 2025) |
| Revenue momentum | 2.32x — December dramatically outperformed prior months |
| Best template | **Main Street Coffee** — highest growth rate & revenue momentum |
| Model to replicate | TABLE-focused, ~3,600+ customer catchment |
| Decision | **GO** — data supports expansion |
| Primary risk | Extreme seasonality (all branches show >3x peak/trough ratio) |

### Staffing Recommendations
| Branch | Afternoon Peak | Evening Peak | Special Note |
|--------|---------------|-------------|---------------|
| Conut Jnah | 2 staff | 2 staff | Highest productivity: 5.1M/staff-hour |
| Main Street Coffee | 2 staff | 2 staff | 3.2M/staff-hour; 6 unique employees Dec |
| Conut - Tyre | 2 staff | 2 staff | Only branch with delivery infrastructure |
| Conut | 2 staff | 2 staff | Declining revenue — monitor closely |

**Key finding:** All branches are currently operating at or near the safety minimum (2 staff). 
**Recommendation:** On peak days (Fri-Sat, December), add 1 additional staff per peak shift at **Conut Jnah** and **Main Street Coffee** given their higher volumes and productivity rates.

### Anomalies to Address
- **9 long-shift entries** (>14 hours) detected across all branches — likely missed punch-outs. System should enforce auto-timeout after 12 hours.
- **Person_0011 (Main Street Coffee)** logged 382 hours in one entry — clear system error requiring immediate correction.

---
## Part 3 — Combo Optimization

In [ ]:
from src.combo_engine import run_combo_analysis

combo_result = run_combo_analysis()
basket_stats  = combo_result['basket_stats']
global_rules  = combo_result['global_rules']
uplift_df     = combo_result['uplift_estimates']
transactions  = combo_result['transactions']

print(f"Transactions parsed : {len(transactions)}")
print(f"Global rules found  : {len(global_rules)}")
print('\nBasket stats:')
print(basket_stats[['n_customers','avg_basket_size','avg_spend_per_order','top_item']])

In [ ]:
# ── Figure 10: Top Association Rules — Lift Bar Chart ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Top 12 global rules by lift
top_rules = global_rules.head(12).copy()
top_rules['rule_label'] = top_rules.apply(
    lambda r: f"{r['antecedent'][:22]}  +\n{r['consequent'][:22]}", axis=1)

ax = axes[0]
bars = ax.barh(top_rules['rule_label'], top_rules['lift'],
               color=COLORS[0], edgecolor='white', linewidth=0.7)
ax.set_xlabel('Lift Score  (>1 = bought together more than by chance)', fontsize=10)
ax.set_title('Top Association Rules — Lift Score\n(Higher = Stronger Co-purchase Signal)',
             fontsize=12, fontweight='bold')
ax.axvline(1.0, color='gray', linestyle='--', alpha=0.5, label='Random baseline (lift=1)')
ax.legend(fontsize=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Right: Confidence vs Lift scatter
ax2 = axes[1]
for _, rule in global_rules.iterrows():
    ax2.scatter(rule['confidence'], rule['lift'], s=rule['support']*3000,
                color=COLORS[2], alpha=0.6, edgecolors='white', linewidths=0.8)

# Annotate top 5
for _, rule in global_rules.head(5).iterrows():
    ax2.annotate(f"{rule['antecedent'][:18]}", (rule['confidence'], rule['lift']),
                 fontsize=7, ha='left', va='bottom',
                 xytext=(4, 2), textcoords='offset points')

ax2.set_xlabel('Confidence  (P(B|A))', fontsize=10)
ax2.set_ylabel('Lift', fontsize=10)
ax2.set_title('Confidence vs Lift\n(Bubble size = Support)', fontsize=12, fontweight='bold')
ax2.axhline(1.0, color='gray', linestyle='--', alpha=0.4)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('combo_rules.png', dpi=150, bbox_inches='tight')
plt.show()
print('Fig 10 saved: combo_rules.png')

In [ ]:
# ── Figure 11: Revenue Uplift Estimates per Combo Deal ─────────────────────
if not uplift_df.empty:
    fig, ax = plt.subplots(figsize=(12, 5))

    x = range(len(uplift_df))
    bars = ax.bar(x, uplift_df['lift'], color=COLORS[2], edgecolor='white',
                  alpha=0.8, label='Lift Score (left)')
    ax.set_ylabel('Lift Score', fontsize=11, color=COLORS[2])
    ax.tick_params(axis='y', labelcolor=COLORS[2])

    ax2 = ax.twinx()
    ax2.plot(x, uplift_df['confidence'], marker='o', color=COLORS[0],
             linewidth=2.5, markersize=8, label='Confidence (right)')
    ax2.set_ylabel('Confidence', fontsize=11, color=COLORS[0])
    ax2.tick_params(axis='y', labelcolor=COLORS[0])

    labels = [f"{r['antecedent'][:20]}\n+{r['consequent'][:20]}"
              for _, r in uplift_df.iterrows()]
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=8, rotation=20, ha='right')
    ax.set_title('Top Combo Deals — Lift & Confidence\n(Best combinations to bundle and promote)',
                 fontsize=13, fontweight='bold')
    ax.spines['top'].set_visible(False)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper right')

    plt.tight_layout()
    plt.savefig('combo_uplift.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Fig 11 saved: combo_uplift.png')
else:
    print('No uplift data available.')

---
## Part 4 — Demand Forecasting

In [ ]:
from src.demand_forecast_engine import run_demand_forecast

forecast_result = run_demand_forecast()
forecasts        = forecast_result['forecasts']
network_monthly  = forecast_result['network_monthly']
branch_share     = forecast_result['branch_share']

print('Network monthly demand:')
print(network_monthly[['month_name','revenue','demand_index','mom_growth_pct']].to_string(index=False))

In [ ]:
# ── Figure 12: Demand Forecast — Historical + 3-Month Projection ────────────
FORECAST_MONTHS = ['Jan 2026', 'Feb 2026', 'Mar 2026']

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for idx, (branch, fc) in enumerate(forecasts.items()):
    ax = axes[idx]
    if 'error' in fc:
        ax.text(0.5, 0.5, fc['error'], ha='center', va='center')
        ax.set_title(branch)
        continue

    hist_x = list(range(len(fc['historical_months'])))
    hist_y = [v / 1e9 for v in fc['historical_revenue']]
    fut_x  = [len(hist_x) + i for i in range(3)]
    fut_y  = [v / 1e9 for v in fc['forecast_ensemble']]
    lo80   = [v / 1e9 for v in fc['lower_80']]
    hi80   = [v / 1e9 for v in fc['upper_80']]

    color = BRANCH_COLORS.get(branch, 'gray')

    # Historical line
    ax.plot(hist_x, hist_y, 'o-', color=color, linewidth=2.5,
            markersize=8, label='Historical', zorder=3)

    # Forecast line (dashed)
    ax.plot([hist_x[-1]] + fut_x, [hist_y[-1]] + fut_y,
            '--o', color=color, linewidth=2, markersize=7,
            alpha=0.85, label='Forecast (ensemble)', zorder=3)

    # 80% CI band
    ax.fill_between([hist_x[-1]] + fut_x,
                    [hist_y[-1]] + lo80,
                    [hist_y[-1]] + hi80,
                    alpha=0.18, color=color, label='80% CI')

    all_x = hist_x + fut_x
    all_labels = [m[:3] for m in fc['historical_months']] + FORECAST_MONTHS
    ax.set_xticks(all_x)
    ax.set_xticklabels(all_labels, fontsize=9, rotation=20)
    ax.axvline(len(hist_x) - 0.5, color='gray', linestyle=':', alpha=0.6)
    ax.text(len(hist_x) - 0.45, ax.get_ylim()[1] * 0.95, 'FORECAST',
            fontsize=8, color='gray', style='italic')

    ax.set_title(f'{branch}\n{fc["trajectory"]} ({fc["monthly_growth_rate_hist"]:+.1f}%/mo)',
                 fontsize=11, fontweight='bold', color=color)
    ax.set_ylabel('Revenue (Billions, scaled)')
    ax.legend(fontsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Demand Forecast by Branch — Aug 2025 History + Jan-Mar 2026 Projection',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('forecast_by_branch.png', dpi=150, bbox_inches='tight')
plt.show()
print('Fig 12 saved: forecast_by_branch.png')

In [ ]:
# ── Figure 13: Branch Revenue Share Evolution (Stacked Area) ────────────────
share_pivot = branch_share.pivot(index='month', columns='branch', values='share_pct').fillna(0)

fig, ax = plt.subplots(figsize=(12, 5))
colors_share = [BRANCH_COLORS.get(b, 'gray') for b in share_pivot.columns]
ax.stackplot(share_pivot.index, share_pivot.T.values,
             labels=share_pivot.columns.tolist(),
             colors=colors_share, alpha=0.85)

month_labels = (branch_share.drop_duplicates('month')
                .sort_values('month')['month_name'].tolist())
ax.set_xticks(share_pivot.index)
ax.set_xticklabels([m[:3] for m in month_labels], fontsize=10)
ax.set_ylabel('Revenue Share (%)')
ax.set_ylim(0, 100)
ax.set_title('Branch Revenue Share Evolution (Aug–Dec 2025)',
             fontsize=13, fontweight='bold')
ax.legend(loc='upper left', fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('forecast_share_evolution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Fig 13 saved: forecast_share_evolution.png')

---
## Part 5 — Coffee & Milkshake Growth Strategy

In [ ]:
from src.coffee_milkshake_engine import run_coffee_milkshake_analysis

cm_result  = run_coffee_milkshake_analysis()
cat_totals = cm_result['category_totals']
opp_matrix = cm_result['opportunity_matrix']
pareto_cat = cm_result['pareto_by_cat']
strategy   = cm_result['strategy_recommendations']

print(f"Total beverage revenue: {cm_result['total_beverage_revenue']:,.0f}")
print('\nCategory breakdown:')
print(cat_totals)

In [ ]:
# ── Figure 14: Beverage Category Revenue Breakdown (Donut + Bar) ────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

CAT_COLORS = {'coffee': '#2E86AB', 'frappe': '#F18F01',
              'milkshake': '#A23B72', 'cold_drink': '#C73E1D'}

# Left: Donut chart
wedge_colors = [CAT_COLORS.get(c, 'gray') for c in cat_totals.index]
wedges, texts, autotexts = axes[0].pie(
    cat_totals.values, labels=cat_totals.index.str.title(),
    autopct='%1.1f%%', colors=wedge_colors, startangle=90,
    wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2),
    textprops=dict(fontsize=11),
)
for at in autotexts:
    at.set_fontsize(10)
    at.set_fontweight('bold')
axes[0].set_title('Beverage Revenue Mix\n(All Branches, 2025)',
                  fontsize=13, fontweight='bold')

# Right: Top 5 items per category grouped bar
top_items_per_cat = {}
for cat, pareto in pareto_cat.items():
    if not pareto.empty:
        top_items_per_cat[cat] = pareto.head(5)

# Build a combined bar chart for top items
all_items = []
for cat, df in top_items_per_cat.items():
    for _, r in df.iterrows():
        all_items.append({'item': r['item'][:22], 'category': cat,
                          'revenue': r['total_revenue']})
items_df = pd.DataFrame(all_items).sort_values('revenue', ascending=True).tail(16)

ax2 = axes[1]
colors_items = [CAT_COLORS.get(r['category'], 'gray') for _, r in items_df.iterrows()]
bars = ax2.barh(items_df['item'], items_df['revenue'] / 1e6,
                color=colors_items, edgecolor='white', linewidth=0.7)
ax2.set_xlabel('Revenue (Millions, scaled units)')
ax2.set_title('Top 16 Beverage SKUs by Revenue\n(Coloured by Category)',
              fontsize=13, fontweight='bold')

legend_patches = [mpatches.Patch(color=c, label=cat.title())
                  for cat, c in CAT_COLORS.items()]
ax2.legend(handles=legend_patches, fontsize=9, loc='lower right')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('coffee_category_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()
print('Fig 14 saved: coffee_category_breakdown.png')

In [ ]:
# ── Figure 15: Growth Opportunity Matrix (Scatter) ──────────────────────────
fig, ax = plt.subplots(figsize=(13, 7))

quad_colors = {'STAR': '#F18F01', 'OPPORTUNITY': '#2E86AB',
               'CASH COW': '#3B1F2B', 'DOG': '#C73E1D'}
quad_markers = {'STAR': '*', 'OPPORTUNITY': '^', 'CASH COW': 'o', 'DOG': 'x'}

qty_med = opp_matrix['total_qty'].median()
rev_med = opp_matrix['rev_per_unit'].median()

for quad, grp in opp_matrix.groupby('quadrant'):
    ax.scatter(grp['total_qty'], grp['rev_per_unit'] / 1e3,
               s=120, label=quad, color=quad_colors.get(quad, 'gray'),
               marker=quad_markers.get(quad, 'o'),
               alpha=0.85, edgecolors='white', linewidths=0.8)

# Annotate OPPORTUNITIES (high value, low volume — most actionable)
for _, r in opp_matrix[opp_matrix['quadrant'] == 'OPPORTUNITY'].iterrows():
    ax.annotate(r['item'][:20], (r['total_qty'], r['rev_per_unit'] / 1e3),
                fontsize=7.5, xytext=(6, 4), textcoords='offset points',
                color='#2E86AB', fontweight='bold')

ax.axvline(qty_med, color='gray', linestyle='--', alpha=0.4)
ax.axhline(rev_med / 1e3, color='gray', linestyle='--', alpha=0.4)
ax.text(qty_med * 1.02, ax.get_ylim()[1] * 0.97, 'STAR', color='#F18F01',
        fontsize=10, fontweight='bold')
ax.text(0, ax.get_ylim()[1] * 0.97, 'OPPORTUNITY', color='#2E86AB',
        fontsize=10, fontweight='bold')
ax.text(qty_med * 1.02, ax.get_ylim()[0] * 1.3 + 0.01, 'CASH COW', color='#3B1F2B',
        fontsize=10, fontweight='bold')
ax.text(0, ax.get_ylim()[0] * 1.3 + 0.01, 'DOG', color='#C73E1D',
        fontsize=10, fontweight='bold')

ax.set_xlabel('Total Quantity Sold (Units)', fontsize=11)
ax.set_ylabel('Revenue per Unit (Thousands, scaled)', fontsize=11)
ax.set_title('Beverage Growth Opportunity Matrix\n'
             '(OPPORTUNITY quadrant = high-value items to promote harder)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='upper right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('coffee_opportunity_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Fig 15 saved: coffee_opportunity_matrix.png')